In [1]:
!uv add requests

Resolved 112 packages in 15ms
Uninstalled 1 package in 14ms
Installed 1 package in 5ms                                  
 - pyyaml==6.0.3
 + pyyaml==5.1


In [2]:
import requests
repo_owner = 'evidentlyai'
repo_name='docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [3]:
len(zip_response.content)

17544837

In [4]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [5]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])  # Print the first 500 characters of the MDX content

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [6]:
!uv add python-frontmatter

Resolved 112 packages in 15ms
Checked 109 packages in 20ms


In [6]:
import frontmatter

post = frontmatter.loads(mdx_content)

In [7]:
post.metadata

{'title': 'Alerts', 'description': 'How to set up alerts.'}

In [8]:
print(post.content[:150])  # Print the first 500 characters of the content

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.
</Check>

![](/images/alerts.png)

To 


In [9]:
_, filename_corrected = filename.split('/', maxsplit=1)
print(filename_corrected)

docs/platform/alerts.mdx


In [10]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [11]:
documents = []
with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith(('.md', '.mdx')):
            continue
        with zip_ref.open(file_path) as file:
            content = file.read().decode('utf-8')
            post = frontmatter.loads(content)
            doc = {
                'content': post.content,
                'title': post.metadata.get('title'),
                'description': post.metadata.get('description'),
                'filename': file_path.split('/', 1)[-1]
            }
            documents.append(doc)

In [12]:
len(documents)


95

In [13]:
!uv add gitsource

Resolved 113 packages in 772ms                                       
Prepared 2 packages in 132ms                                                 gitsource            ------------------------------ 8.77 KiB/8.77 KiB           
Uninstalled 1 package in 2ms
Installed 2 packages in 3mscamp==0.1.0 (from file:///Users/P
 ~ ai-engineering-buildcamp==0.1.0 (from file:///Users/Paddu/ai-engineering-buildcamp)
 + gitsource==0.0.5


In [19]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [20]:
md_file = files[10]

In [ ]:
documents = [f.parse() for f in files]

In [21]:
len(documents)

95

In [22]:
documents[10]

{'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python\nmy_report.save_html(“file.html”)\n```\n\nThis option is useful for sharing Reports 

In [23]:
!uv add minsearch

Resolved 123 packages in 807ms                                       
Prepared 11 packages in 1.15s                                                scipy                ------------------------------ 17.76 MiB/19.52 MiB         scipy                ------------------------------ 14.20 MiB/19.52 MiB         scipy                ------------------------------ 8.99 MiB/19.52 MiB          scipy                ------------------------------ 6.61 MiB/19.52 MiB          scipy                ------------------------------ 1.92 MiB/19.52 MiB          scipy                ------------------------------ 1.07 MiB/19.52 MiB          scipy                ------------------------------ 493.72 KiB/19.52 MiB        
Uninstalled 1 package in 2ms
Installed 11 packages in 132msamp==0.1.0 (from file:///Users
 ~ ai-engineering-buildcamp==0.1.0 (from file:///Users/Paddu/ai-engineering-buildcamp)
 + cloudpickle==3.1.2
 + joblib==1.6.0
 + minsearch==0.2.0
 + narwhals==2.26.0
 + numpy==2.5.3
 + pandas==3.0.6
 + 

In [24]:
from minsearch import Index

In [25]:
index = Index(
    text_fields=["title", "description","content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [26]:
query = 'LLM as a Judge'
results = index.search(query, num_results=5)

In [27]:
len(results)

5

In [28]:
len(results[0]['content'])

21834

In [29]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [30]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})
        
        start = end - step
        
        if end >= text_length:
            break
    
    return chunks

In [32]:
document_chunks =[]

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')
    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)


In [33]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [34]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [35]:
results = chunk_index.search(query)

In [36]:
results

[{'start': 0,
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different cont

In [37]:
from gitsource import chunk_documents

In [38]:
document_chunks = chunk_documents(documents, size=3000, step=1500) 

In [39]:
from openai import OpenAI
openai_client = OpenAI()

In [40]:
search_result = chunk_index.search(query, num_results=5)

In [41]:
query = 'how do i implement llm as a judge?'

In [42]:
import json

In [43]:
search_result_json = json.dumps(search_result, indent=2)

In [45]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [46]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [47]:
answer = llm(user_prompt, instructions)
print(answer)

To implement an LLM (Language Model) as a judge, follow these steps based on the provided context:

1. **Set Up Your Environment**:
   - **Install Necessary Libraries**:
     ```bash
     pip install evidently
     ```
   - **Import Required Modules**:
     ```python
     import pandas as pd
     import numpy as np
     from evidently import Dataset, DataDefinition, Report, BinaryClassification
     from evidently.llm.templates import BinaryClassificationPromptTemplate
     ```

2. **Pass Your OpenAI API Key**:
   Set your OpenAI API key as an environment variable:
   ```python
   import os
   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
   ```

3. **Create Your Dataset**:
   Construct a dataset consisting of:
   - **Questions**: User inputs.
   - **Target Responses**: Approved, correct replies.
   - **New Responses**: Responses generated by the system.
   - **Manual Labels**: For evaluating correctness.
   You can generate the data directly in code as shown in the context.

4. **Design t

In [48]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [49]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_result, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [50]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [51]:
rag('how do I implement llm as a judge?')

'To implement an LLM as a judge, follow these steps based on the provided context:\n\n1. **Setup Requirements**:\n   - Ensure you have basic knowledge of Python.\n   - Obtain an OpenAI API key for using the LLM.\n\n2. **Installation**:\n   - Install the required library:\n     ```python\n     pip install evidently\n     ```\n\n3. **Import Required Modules**:\n   ```python\n   import pandas as pd\n   import numpy as np\n   from evidently import Dataset, DataDefinition, Report\n   from evidently.presets import TextEvals, ClassificationPreset\n   from evidently.llm.templates import BinaryClassificationPromptTemplate\n   ```\n\n4. **Set your OpenAI API Key**:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   ```\n\n5. **Create an Evaluation Dataset**:\n   - Construct a dataset including questions, target responses, and new responses.\n   - Label your data to define correct responses and understand the LLM\'s performance.\n\n6. **Design LLM Evaluator Prompts**:\n